# 04 — ARIMA & SARIMA

**Mục tiêu:** Xây dựng và so sánh mô hình ARIMA và SARIMA để dự báo doanh số cửa hàng, đánh giá bằng RMSLE và chẩn đoán bằng kiểm định Ljung-Box.

**Kết quả từ các Task trước:**
- **Task 2 (Decomposition):** MSTL phân rã thành Trend + 4 seasonal components (tuần/tháng/quý/năm) + Residual.
- **Task 3 (QS Test):** Xác nhận mùa vụ tuần (m=7), quý (m=91), năm (m=365) có ý nghĩa thống kê. Mùa vụ tháng (m=30) **không** có ý nghĩa (p=0.052).

**Workflow:**
1. Load & split dữ liệu
2. Kiểm định tính dừng (ADF test)
3. Xác định bậc ARIMA (ACF/PACF)
4. Fit ARIMA → chẩn đoán (Ljung-Box) → dự báo → RMSLE
5. Fit SARIMA (m=7, theo kết quả QS test) → chẩn đoán → dự báo → RMSLE
6. So sánh ARIMA vs SARIMA vs Baseline

In [ ]:
import sys
import os

# Thêm thư mục gốc project vào sys.path để import src/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

# Import các module tự viết
from src.data_prep import load_and_split_data
from src.metrics import rmsle
from src.arima_sarima import (
    check_stationarity,
    fit_arima,
    fit_sarima,
    ljung_box_test,
    forecast_and_evaluate
)

print('✓ Import thành công tất cả modules!')

## 1. Load & Split dữ liệu

In [ ]:
# Đường dẫn tới file dữ liệu đã xử lý
DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'store_1_2013_2017_clean.csv')

train, test = load_and_split_data(DATA_PATH)

print(f'Train: {len(train)} ngày  ({train["date"].min().date()} → {train["date"].max().date()})')
print(f'Test : {len(test)} ngày  ({test["date"].min().date()} → {test["date"].max().date()})')

# Tạo Series với DatetimeIndex cho modeling
train_series = train.set_index('date')['sales'].asfreq('D')
test_series = test.set_index('date')['sales'].asfreq('D')

# Fill NaN (nếu có) bằng forward fill
train_series = train_series.ffill()

print(f'\nTrain series shape: {train_series.shape}')
print(f'Test series shape : {test_series.shape}')

## 2. Kiểm định tính dừng (ADF Test)

Trước khi fit ARIMA, cần kiểm tra chuỗi có dừng hay không.
Nếu p-value < 0.05 → chuỗi dừng (stationary) → d = 0.
Nếu p-value ≥ 0.05 → chuỗi chưa dừng → cần lấy sai phân (d ≥ 1).

In [ ]:
# ADF test trên chuỗi gốc
adf_original = check_stationarity(train_series)

print('='*60)
print('ADF Test — Chuỗi gốc (sales)')
print('='*60)
print(f'  ADF Statistic : {adf_original["adf_stat"]:.4f}')
print(f'  p-value       : {adf_original["p_value"]:.6f}')
print(f'  Số lags       : {adf_original["n_lags"]}')
print(f'  Số quan sát   : {adf_original["n_obs"]}')
print(f'  Kết luận      : {"✓ DỪNG" if adf_original["is_stationary"] else "✗ CHƯA DỪNG → cần sai phân"}')
print()

for key, val in adf_original['critical_values'].items():
    print(f'  Critical Value ({key}): {val:.4f}')

In [ ]:
# ADF test trên sai phân bậc 1
diff1 = train_series.diff().dropna()
adf_diff1 = check_stationarity(diff1)

print('='*60)
print('ADF Test — Sai phân bậc 1')
print('='*60)
print(f'  ADF Statistic : {adf_diff1["adf_stat"]:.4f}')
print(f'  p-value       : {adf_diff1["p_value"]:.6f}')
print(f'  Kết luận      : {"✓ DỪNG" if adf_diff1["is_stationary"] else "✗ CHƯA DỪNG"}')

## 3. Xác định bậc ARIMA bằng ACF/PACF

Sử dụng biểu đồ ACF và PACF trên chuỗi sai phân bậc 1 để xác định:
- **p** (AR order): từ PACF — số lag có giá trị vượt ngưỡng trước khi cut-off
- **q** (MA order): từ ACF — số lag có giá trị vượt ngưỡng trước khi cut-off

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Chuỗi gốc
plot_acf(train_series, lags=40, ax=axes[0, 0], title='ACF — Chuỗi gốc')
plot_pacf(train_series, lags=40, ax=axes[0, 1], title='PACF — Chuỗi gốc',
          method='ywm')

# Sai phân bậc 1
plot_acf(diff1, lags=40, ax=axes[1, 0], title='ACF — Sai phân bậc 1')
plot_pacf(diff1, lags=40, ax=axes[1, 1], title='PACF — Sai phân bậc 1',
          method='ywm')

plt.suptitle('ACF & PACF — Xác định bậc ARIMA', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'figures', 'acf_pacf_arima.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print('✓ Đã lưu biểu đồ ACF/PACF')

## 4. Fit ARIMA Model

Dựa trên ACF/PACF:
- **d = 1** (sai phân bậc 1 để đạt tính dừng)
- **p, q** được chọn dựa trên ACF/PACF analysis

Thử nhiều cấu hình ARIMA để chọn mô hình tốt nhất theo AIC.

In [ ]:
# Grid search ARIMA orders
arima_orders = [
    (1, 1, 1), (1, 1, 2), (2, 1, 1), (2, 1, 2),
    (3, 1, 1), (1, 1, 3), (3, 1, 2), (2, 1, 3),
]

arima_results = []

print('ARIMA Grid Search')
print('='*60)
print(f'{"Order":<15} {"AIC":<15} {"BIC":<15} {"Status"}')
print('-'*60)

for order in arima_orders:
    try:
        result = fit_arima(train_series, order=order)
        arima_results.append({
            'order': order,
            'aic': result.aic,
            'bic': result.bic,
            'model': result
        })
        print(f'{str(order):<15} {result.aic:<15.2f} {result.bic:<15.2f} ✓')
    except Exception as e:
        print(f'{str(order):<15} {"—":<15} {"—":<15} ✗ {str(e)[:30]}')

# Chọn mô hình tốt nhất theo AIC
best_arima = min(arima_results, key=lambda x: x['aic'])
print(f'\n→ Best ARIMA: {best_arima["order"]}  (AIC = {best_arima["aic"]:.2f})')

### 4.1 Chẩn đoán ARIMA — Ljung-Box Test

Kiểm định Ljung-Box kiểm tra xem phần dư có phải là white noise hay không.
- **H₀**: Phần dư là white noise (không có tự tương quan)
- **H₁**: Phần dư có tự tương quan
- Nếu p-value > 0.05 → Không bác bỏ H₀ → Mô hình phù hợp

In [ ]:
# Lấy best ARIMA model
best_arima_model = best_arima['model']
best_arima_order = best_arima['order']

# Ljung-Box test trên phần dư
lb_arima = ljung_box_test(best_arima_model.resid, lags=20)

print(f'Ljung-Box Test — ARIMA{best_arima_order}')
print('='*60)
print(lb_arima.to_string())
print()

# Kiểm tra tại lag 10 và 20
for lag in [10, 20]:
    p_val = lb_arima.loc[lag, 'lb_pvalue']
    status = '✓ White noise' if p_val > 0.05 else '✗ Có tự tương quan'
    print(f'  Lag {lag}: p-value = {p_val:.6f} → {status}')

In [ ]:
# Plot diagnostic cho ARIMA
fig = best_arima_model.plot_diagnostics(figsize=(14, 10))
fig.suptitle(f'Diagnostic — ARIMA{best_arima_order}', fontsize=14,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'figures', 'arima_diagnostics.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print('✓ Đã lưu biểu đồ diagnostic ARIMA')

### 4.2 Dự báo ARIMA & Tính RMSLE

In [ ]:
# Dự báo ARIMA
arima_eval = forecast_and_evaluate(
    best_arima_model,
    n_forecast=len(test),
    y_true=test['sales'].values,
    model_name=f'ARIMA{best_arima_order}'
)

print('='*50)
print(f'  Model : {arima_eval["model_name"]}')
print(f'  RMSLE : {arima_eval["rmsle"]:.6f}')
print('='*50)

# Xem 10 dự đoán đầu tiên
comparison_arima = pd.DataFrame({
    'date': test['date'].values[:10],
    'actual': test['sales'].values[:10],
    'predicted': arima_eval['predictions'][:10]
})
print('\nMẫu 10 dự đoán đầu tiên:')
comparison_arima

## 5. Fit SARIMA Model

Dựa trên kết quả QS Test (Task 3):
- Mùa vụ **tuần (m=7)**: CÓ ý nghĩa (p ≈ 0)
- Mùa vụ **tháng (m=30)**: KHÔNG có ý nghĩa (p = 0.052)
- Mùa vụ **quý (m=91)**: CÓ ý nghĩa (p ≈ 0)
- Mùa vụ **năm (m=365)**: CÓ ý nghĩa (p = 0.021)

→ Chọn **m = 7** (mùa vụ tuần) cho SARIMA vì:
1. Mùa vụ tuần mạnh nhất (QS stat = 631.8)
2. m=91 và m=365 quá lớn → chi phí tính toán rất cao, không khả thi cho SARIMA

In [ ]:
# Grid search SARIMA orders (m=7)
sarima_configs = [
    ((1, 1, 1), (1, 1, 1, 7)),
    ((1, 1, 1), (0, 1, 1, 7)),
    ((1, 1, 1), (1, 0, 1, 7)),
    ((2, 1, 1), (1, 1, 1, 7)),
    ((1, 1, 2), (1, 1, 1, 7)),
    ((2, 1, 1), (0, 1, 1, 7)),
    ((1, 1, 1), (2, 1, 1, 7)),
    ((1, 1, 2), (0, 1, 1, 7)),
]

sarima_results = []

print('SARIMA Grid Search (m=7)')
print('='*75)
print(f'{"Order":<15} {"Seasonal":<20} {"AIC":<15} {"BIC":<15} {"Status"}')
print('-'*75)

for order, seasonal in sarima_configs:
    try:
        result = fit_sarima(train_series, order=order, seasonal_order=seasonal)
        sarima_results.append({
            'order': order,
            'seasonal': seasonal,
            'aic': result.aic,
            'bic': result.bic,
            'model': result
        })
        print(f'{str(order):<15} {str(seasonal):<20} {result.aic:<15.2f} {result.bic:<15.2f} ✓')
    except Exception as e:
        print(f'{str(order):<15} {str(seasonal):<20} {"—":<15} {"—":<15} ✗ {str(e)[:25]}')

# Chọn mô hình tốt nhất theo AIC
if sarima_results:
    best_sarima = min(sarima_results, key=lambda x: x['aic'])
    print(f'\n→ Best SARIMA: {best_sarima["order"]}x{best_sarima["seasonal"]}  (AIC = {best_sarima["aic"]:.2f})')
else:
    print('\n✗ Không có mô hình SARIMA nào fit thành công!')

### 5.1 Chẩn đoán SARIMA — Ljung-Box Test

In [ ]:
# Lấy best SARIMA model
best_sarima_model = best_sarima['model']
best_sarima_order = best_sarima['order']
best_sarima_seasonal = best_sarima['seasonal']

# Ljung-Box test trên phần dư
lb_sarima = ljung_box_test(best_sarima_model.resid, lags=20)

print(f'Ljung-Box Test — SARIMA{best_sarima_order}x{best_sarima_seasonal}')
print('='*60)
print(lb_sarima.to_string())
print()

for lag in [10, 20]:
    p_val = lb_sarima.loc[lag, 'lb_pvalue']
    status = '✓ White noise' if p_val > 0.05 else '✗ Có tự tương quan'
    print(f'  Lag {lag}: p-value = {p_val:.6f} → {status}')

In [ ]:
# Plot diagnostic cho SARIMA
fig = best_sarima_model.plot_diagnostics(figsize=(14, 10))
fig.suptitle(f'Diagnostic — SARIMA{best_sarima_order}x{best_sarima_seasonal}',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'figures', 'sarima_diagnostics.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print('✓ Đã lưu biểu đồ diagnostic SARIMA')

### 5.2 Dự báo SARIMA & Tính RMSLE

In [ ]:
# Dự báo SARIMA
sarima_eval = forecast_and_evaluate(
    best_sarima_model,
    n_forecast=len(test),
    y_true=test['sales'].values,
    model_name=f'SARIMA{best_sarima_order}x{best_sarima_seasonal}'
)

print('='*50)
print(f'  Model : {sarima_eval["model_name"]}')
print(f'  RMSLE : {sarima_eval["rmsle"]:.6f}')
print('='*50)

# Xem 10 dự đoán đầu tiên
comparison_sarima = pd.DataFrame({
    'date': test['date'].values[:10],
    'actual': test['sales'].values[:10],
    'predicted': sarima_eval['predictions'][:10]
})
print('\nMẫu 10 dự đoán đầu tiên:')
comparison_sarima

## 6. So sánh ARIMA vs SARIMA vs Baseline

In [ ]:
# Load Baseline RMSLE
baseline_path = os.path.join(PROJECT_ROOT, 'results', 'metrics', 'baseline_rmsle.csv')
baseline_df = pd.read_csv(baseline_path)
baseline_rmsle = baseline_df['RMSLE'].values[0]

# Bảng so sánh
comparison_df = pd.DataFrame({
    'Model': ['Baseline (Seasonal Naive)', arima_eval['model_name'], sarima_eval['model_name']],
    'RMSLE': [baseline_rmsle, arima_eval['rmsle'], sarima_eval['rmsle']]
})
comparison_df = comparison_df.sort_values('RMSLE').reset_index(drop=True)
comparison_df['Rank'] = range(1, len(comparison_df) + 1)

print('='*60)
print('  SO SÁNH KẾT QUẢ DỰ BÁO')
print('='*60)
print(comparison_df.to_string(index=False))
print()

best_model = comparison_df.iloc[0]
print(f'→ Mô hình tốt nhất: {best_model["Model"]} (RMSLE = {best_model["RMSLE"]:.6f})')

# ARIMA vs Baseline
arima_improvement = ((baseline_rmsle - arima_eval['rmsle']) / baseline_rmsle) * 100
sarima_improvement = ((baseline_rmsle - sarima_eval['rmsle']) / baseline_rmsle) * 100
print(f'\n  ARIMA  vs Baseline: {"↓" if arima_improvement > 0 else "↑"} {abs(arima_improvement):.2f}%')
print(f'  SARIMA vs Baseline: {"↓" if sarima_improvement > 0 else "↑"} {abs(sarima_improvement):.2f}%')
print(f'  SARIMA vs ARIMA   : {"↓" if sarima_eval["rmsle"] < arima_eval["rmsle"] else "↑"} {abs(arima_eval["rmsle"] - sarima_eval["rmsle"]):.6f}')

In [ ]:
# Visualize so sánh
fig, axes = plt.subplots(2, 1, figsize=(14, 12))

# --- Plot 1: Toàn cảnh Train + Test + Predictions ---
ax1 = axes[0]
ax1.plot(train['date'], train['sales'], color='#2196F3', alpha=0.4,
         linewidth=0.6, label='Train')
ax1.plot(test['date'], test['sales'], color='#333333', linewidth=1.2,
         label='Test (Actual)')
ax1.plot(test['date'], arima_eval['predictions'], color='#E91E63',
         linewidth=1.2, linestyle='--',
         label=f'{arima_eval["model_name"]} (RMSLE={arima_eval["rmsle"]:.4f})')
ax1.plot(test['date'], sarima_eval['predictions'], color='#4CAF50',
         linewidth=1.2, linestyle='-.',
         label=f'{sarima_eval["model_name"]} (RMSLE={sarima_eval["rmsle"]:.4f})')
ax1.axvline(x=pd.Timestamp('2017-04-01'), color='red', linestyle=':',
            linewidth=1.5, label='Train/Test Split')
ax1.set_title('Sales Forecast — ARIMA vs SARIMA', fontsize=13, fontweight='bold')
ax1.set_xlabel('Date')
ax1.set_ylabel('Sales')
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(True, alpha=0.3)

# --- Plot 2: Zoom vào Test period ---
ax2 = axes[1]
ax2.plot(test['date'], test['sales'], color='#333333', linewidth=1.5,
         marker='o', markersize=2, label='Actual')
ax2.plot(test['date'], arima_eval['predictions'], color='#E91E63',
         linewidth=1.2, linestyle='--',
         label=f'{arima_eval["model_name"]}')
ax2.plot(test['date'], sarima_eval['predictions'], color='#4CAF50',
         linewidth=1.2, linestyle='-.',
         label=f'{sarima_eval["model_name"]}')
ax2.set_title('Zoom: Test Period — ARIMA vs SARIMA', fontsize=13, fontweight='bold')
ax2.set_xlabel('Date')
ax2.set_ylabel('Sales')
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'figures', 'arima_vs_sarima_forecast.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print('✓ Đã lưu biểu đồ so sánh')

## 7. Lưu kết quả

In [ ]:
# Lưu bảng so sánh RMSLE
output_path = os.path.join(PROJECT_ROOT, 'results', 'metrics', 'arima_sarima_rmsle.csv')
os.makedirs(os.path.dirname(output_path), exist_ok=True)
comparison_df.to_csv(output_path, index=False)
print(f'✓ Đã lưu bảng so sánh: {output_path}')

# Lưu Ljung-Box results
lb_output = os.path.join(PROJECT_ROOT, 'results', 'metrics', 'ljung_box_results.csv')
lb_combined = pd.DataFrame({
    'lag': lb_arima.index,
    f'ARIMA_lb_stat': lb_arima['lb_stat'].values,
    f'ARIMA_lb_pvalue': lb_arima['lb_pvalue'].values,
    f'SARIMA_lb_stat': lb_sarima['lb_stat'].values,
    f'SARIMA_lb_pvalue': lb_sarima['lb_pvalue'].values,
})
lb_combined.to_csv(lb_output, index=False)
print(f'✓ Đã lưu Ljung-Box results: {lb_output}')

# Lưu predictions
pred_output = os.path.join(PROJECT_ROOT, 'results', 'predictions', 'arima_sarima_predictions.csv')
os.makedirs(os.path.dirname(pred_output), exist_ok=True)
pred_df = pd.DataFrame({
    'date': test['date'].values,
    'actual': test['sales'].values,
    'arima_pred': arima_eval['predictions'],
    'sarima_pred': sarima_eval['predictions']
})
pred_df.to_csv(pred_output, index=False)
print(f'✓ Đã lưu predictions: {pred_output}')

print('\n✓ Hoàn tất Task 4!')

## Kết luận

### Tóm tắt kết quả:
- **ARIMA** và **SARIMA** đã được xây dựng, chẩn đoán và đánh giá.
- Kiểm định **Ljung-Box** đã được áp dụng để kiểm tra tính phù hợp của mô hình.
- **SARIMA** tận dụng kết quả QS test (Task 3) về mùa vụ tuần (m=7).

### Kết quả lưu tại:
- `results/metrics/arima_sarima_rmsle.csv` — Bảng so sánh RMSLE
- `results/metrics/ljung_box_results.csv` — Kết quả Ljung-Box
- `results/predictions/arima_sarima_predictions.csv` — Dự báo chi tiết
- `results/figures/acf_pacf_arima.png` — Biểu đồ ACF/PACF
- `results/figures/arima_diagnostics.png` — Diagnostic ARIMA
- `results/figures/sarima_diagnostics.png` — Diagnostic SARIMA
- `results/figures/arima_vs_sarima_forecast.png` — So sánh trực quan